In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Disable GPU usage (optional)
os.environ["TF_DISABLE"] = "1"  # Attempt to disable TensorFlow (not always effective)

# tensorflow is diabled because wo pytorch ke g*nd mai ungli kr rha tha 
# Isiliye bhaga dia usse "haramkhor"

In [5]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import matplotlib.pyplot as plt
import seaborn as sns

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\IQBAL
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\IQBAL
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
# # Load your dashboard data
# dashboard_data = pd.read_csv("data/raw/dashboard_data.csv")

# # Load Bitext dataset
# bitext_data = pd.read_csv("data/raw/Bitext_Sample_Customer_Service_Training_Dataset.csv")

# # Prepare dashboard data for intent classification
# dashboard_data['intent'] = dashboard_data['status'].map({
#     'open': 'ticket_query',
#     'scheduled': 'scheduled_query',
#     'resolved': 'resolved_query'
# })

# # Select relevant columns from dashboard_data
# dashboard_intents = dashboard_data[['ticket_type', 'intent']].rename(columns={'ticket_type': 'text'})
# dashboard_intents = dashboard_intents.dropna()

# # Debug: Check dashboard intents
# print("Dashboard intents distribution before deduplication:")
# print(dashboard_intents['intent'].value_counts())

# # Select relevant columns from Bitext dataset
# bitext_intents = bitext_data[['utterance', 'intent']].rename(columns={'utterance': 'text'})
# bitext_intents = bitext_intents[bitext_intents['intent'].isin(['cancel_order', 'feedback', 'order'])]
# bitext_intents = bitext_intents.dropna()

# # Debug: Check Bitext intents
# print("Bitext intents distribution before deduplication:")
# print(bitext_intents['intent'].value_counts())

# # Combine the datasets
# labeled_data = pd.concat([dashboard_intents, bitext_intents], axis=0, ignore_index=True)

# # Remove duplicate text entries
# labeled_data = labeled_data.drop_duplicates(subset=['text'])

# # Debug: Check combined distribution after deduplication
# print("Combined intents distribution after deduplication:")
# print(labeled_data['intent'].value_counts())

# # Balance the dataset
# # Target number of samples per intent (use the minimum as a baseline, or set a reasonable value)
# min_samples = 50  # Adjust this based on your needs
# balanced_data = pd.DataFrame()

# for intent in labeled_data['intent'].unique():
#     intent_data = labeled_data[labeled_data['intent'] == intent]
#     if len(intent_data) > min_samples:
#         # Undersample if too many samples
#         intent_data = intent_data.sample(min_samples, random_state=42)
#     elif len(intent_data) < min_samples:
#         # Oversample if too few samples
#         intent_data = intent_data.sample(min_samples, replace=True, random_state=42)
#     balanced_data = pd.concat([balanced_data, intent_data])

# # Shuffle the balanced dataset
# labeled_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

# # Debug: Check balanced distribution
# print("Balanced intents distribution:")
# print(labeled_data['intent'].value_counts())

# # Create intent mappings
# intents = labeled_data['intent'].unique().tolist()
# intent_to_label = {intent: idx for idx, intent in enumerate(intents)}
# label_to_intent = {idx: intent for intent, idx in intent_to_label.items()}

# # Add labels to the dataset
# labeled_data['label'] = labeled_data['intent'].map(intent_to_label)

# # Split the dataset with stratification to preserve intent distribution
# train_data, val_data = train_test_split(labeled_data, test_size=0.2, stratify=labeled_data['intent'], random_state=42)

# # Extract texts and labels
# train_texts = train_data['text'].values
# train_labels = train_data['label'].values
# val_texts = val_data['text'].values
# val_labels = val_data['label'].values

# # Verify no overlap
# train_set = set(train_texts)
# val_set = set(val_texts)
# overlap = train_set.intersection(val_set)
# print("Overlap between train and val sets:", overlap)
# print("Train set size:", len(train_texts))
# print("Val set size:", len(val_texts))

# # Check intent distribution in train and val sets
# print("Train set intent distribution:")
# print(train_data['intent'].value_counts())
# print("Val set intent distribution:")
# print(val_data['intent'].value_counts())

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Step 1: Generate Synthetic Dashboard Data
synthetic_data = {
    'text': [
        "My solar inverter is not working.",
        "I have an issue with my solar panel.",
        "The battery backup is failing.",
        "Facing trouble with system monitoring.",
        "My inverter shuts down randomly.",
        "System shows error message.",
        "Power generation has stopped.",
        "Panels are not charging properly.",
        "Issue with grid connectivity.",
        "Unable to start the inverter.",
        "When is my maintenance scheduled?",
        "I have an upcoming service appointment.",
        "When will the technician visit?",
        "Is my inverter service scheduled?",
        "Please confirm my service date.",
        "Scheduled maintenance reminder needed.",
        "When is the cleaning scheduled?",
        "Technician visit for system check scheduled?",
        "Battery inspection date?",
        "Panel cleaning appointment confirmation?",
        "Thanks, my inverter issue is resolved.",
        "Technician fixed the solar panel problem.",
        "Issue has been resolved successfully.",
        "My system is now working fine.",
        "Thanks for repairing the inverter.",
        "Panel cleaning was completed successfully.",
        "No more battery backup issues.",
        "Service team fixed the monitoring error.",
        "System is operating perfectly now.",
        "Resolved the power generation problem."
    ] * 10,  # Replicating to get 300 samples
    'intent': (
        ["ticket_query"] * 10 +
        ["scheduled_query"] * 10 +
        ["resolved_query"] * 10
    ) * 10
}

dashboard_intents = pd.DataFrame(synthetic_data)

#  Debug: Check dashboard intents
print("Dashboard intents distribution before deduplication:")
print(dashboard_intents['intent'].value_counts())

# 🚀 Step 2: Load Bitext Dataset
bitext_data = pd.read_csv("data/raw/Bitext_Sample_Customer_Service_Training_Dataset.csv")

# Select relevant columns from Bitext dataset
bitext_intents = bitext_data[['utterance', 'intent']].rename(columns={'utterance': 'text'})
bitext_intents = bitext_intents[bitext_intents['intent'].isin(['cancel_order', 'feedback', 'order'])]
bitext_intents = bitext_intents.dropna()

#  Debug: Check Bitext intents
print("Bitext intents distribution before deduplication:")
print(bitext_intents['intent'].value_counts())

#  Step 3: Combine the datasets
labeled_data = pd.concat([dashboard_intents, bitext_intents], axis=0, ignore_index=True)

# Remove duplicate text entries
labeled_data = labeled_data.drop_duplicates(subset=['text'])

#  Debug: Check combined distribution after deduplication
print("Combined intents distribution after deduplication:")
print(labeled_data['intent'].value_counts())

#  Step 4: Balance the dataset
min_samples = 300  # Bigger now because we have more data
balanced_data = pd.DataFrame()

for intent in labeled_data['intent'].unique():
    intent_data = labeled_data[labeled_data['intent'] == intent]
    if len(intent_data) > min_samples:
        # Undersample
        intent_data = intent_data.sample(min_samples, random_state=42)
    elif len(intent_data) < min_samples:
        # Oversample
        intent_data = intent_data.sample(min_samples, replace=True, random_state=42)
    balanced_data = pd.concat([balanced_data, intent_data])

# Shuffle the balanced dataset
labeled_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

#  Debug: Check balanced distribution
print("Balanced intents distribution:")
print(labeled_data['intent'].value_counts())

#  Step 5: Create intent mappings
intents = labeled_data['intent'].unique().tolist()
intent_to_label = {intent: idx for idx, intent in enumerate(intents)}
label_to_intent = {idx: intent for intent, idx in intent_to_label.items()}

# Add labels to the dataset
labeled_data['label'] = labeled_data['intent'].map(intent_to_label)

# Step 6: Split the dataset
train_data, val_data = train_test_split(labeled_data, test_size=0.2, stratify=labeled_data['intent'], random_state=42)

# Extract texts and labels
train_texts = train_data['text'].values
train_labels = train_data['label'].values
val_texts = val_data['text'].values
val_labels = val_data['label'].values

#  Step 7: Verify splits
train_set = set(train_texts)
val_set = set(val_texts)
overlap = train_set.intersection(val_set)
print("Overlap between train and val sets:", overlap)
print("Train set size:", len(train_texts))
print("Val set size:", len(val_texts))

# Check intent distribution in train and val sets
print("Train set intent distribution:")
print(train_data['intent'].value_counts())
print("Val set intent distribution:")
print(val_data['intent'].value_counts())


Dashboard intents distribution before deduplication:
intent
ticket_query       100
scheduled_query    100
resolved_query     100
Name: count, dtype: int64
Bitext intents distribution before deduplication:
intent
cancel_order    305
Name: count, dtype: int64
Combined intents distribution after deduplication:
intent
cancel_order       305
ticket_query        10
scheduled_query     10
resolved_query      10
Name: count, dtype: int64
Balanced intents distribution:
intent
cancel_order       300
resolved_query     300
ticket_query       300
scheduled_query    300
Name: count, dtype: int64
Overlap between train and val sets: {'Technician fixed the solar panel problem.', 'Please confirm my service date.', 'My solar inverter is not working.', 'Panel cleaning appointment confirmation?', 'Facing trouble with system monitoring.', 'The battery backup is failing.', 'Thanks, my inverter issue is resolved.', 'I have an issue with my solar panel.', 'Scheduled maintenance reminder needed.', 'Panels are 

In [8]:
class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load the tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Create Dataset objects
train_dataset = IntentDataset(train_texts, train_labels, tokenizer)
val_dataset = IntentDataset(val_texts, val_labels, tokenizer)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
# Load the model
num_labels = len(intents)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)

# Move model to device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training loop
num_epochs = 3
for epoch in range(num_epochs):
    # Training
    model.train()
    total_train_loss = 0
    total_train_correct = 0
    total_train_samples = 0

    for batch in train_loader:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Compute training loss and accuracy
        total_train_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        total_train_correct += (predictions == labels).sum().item()
        total_train_samples += labels.size(0)

    avg_train_loss = total_train_loss / len(train_loader)
    train_accuracy = total_train_correct / total_train_samples
    train_losses.append(avg_train_loss)
    train_accuracies.append(train_accuracy)

    # Validation
    model.eval()
    total_val_loss = 0
    total_val_correct = 0
    total_val_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            # Compute validation loss and accuracy
            total_val_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            total_val_correct += (predictions == labels).sum().item()
            total_val_samples += labels.size(0)

            # Store predictions and labels for confusion matrix
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = total_val_correct / total_val_samples
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_accuracy)

    # Print metrics
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

# Save the fine-tuned model
model.save_pretrained("models/distilbert-intent-classifier")
tokenizer.save_pretrained("models/distilbert-intent-classifier")

# Plot training graph
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accuracies, label='Train Accuracy')
plt.plot(epochs, val_accuracies, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=intents, yticklabels=intents)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [9]:
def recognize_intent(user_input):
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_label = torch.argmax(outputs.logits, dim=1).item()
    
    predicted_intent = label_to_intent[predicted_label]
    return predicted_intent

# Test intent recognition
# test_input = "I need help with my inverter not functioning"
test_input = "Issue with grid connectivity."
intent = recognize_intent(test_input)
print(f"Detected intent: {intent}")

NameError: name 'device' is not defined

In [3]:
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return ' '.join(tokens)

dashboard_data['processed_text'] = dashboard_data['ticket_type'].apply(preprocess_text)

def generate_response(intent, user_input):
    processed_input = preprocess_text(user_input)
    matching_row = dashboard_data[dashboard_data['processed_text'].str.contains(processed_input, na=False)]

    if intent == 'cancel_order':
        response = "To cancel your order, please log into our portal and navigate to the 'Cancel Order' section."
    elif intent == 'ticket_query':
        response = f"We've noted your issue: {user_input}. A technician will assist you soon."
    elif intent == 'scheduled_query':
        if not matching_row.empty:
            scheduled_date = matching_row.iloc[0]['scheduled_date']
            response = f"Your {user_input} is scheduled for {scheduled_date}. We’ll send a reminder closer to the date."
        else:
            response = "It looks like you have a scheduled service, but I don’t have the details. Please check the portal."
    elif intent == 'resolved_query':
        response = f"Your issue ({user_input}) has already been resolved. If you have further questions, please let us know!"
    elif intent == 'feedback':
        response = "Thank you for your feedback! We’ll pass it along to our team to improve our services."
    elif intent == 'order':
        response = "To place an order, please visit our website or contact our support team at support@solarenergy.com."
    else:
        response = "I'm sorry, I didn’t quite understand that. Could you please rephrase your request?"
    
    return response

response = generate_response(intent, test_input)
print(f"Response: {response}")

NameError: name 'dashboard_data' is not defined

In [73]:
def chatbot():
    print("Welcome to AUITS Support AI! Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        
        if user_input.lower() in ['hello', 'hi', 'hey', 'how are you?', 'how are you']:
            print("Hello! How can I assist you?")
            continue  # Skip intent detection on greetings

        intent = recognize_intent(user_input)
        response = generate_response(intent, user_input)
        print(f"Bot: {response}")

chatbot()

Welcome to AUITS Support AI! Type 'exit' to quit.


You:  ihave with my inverster


Bot: To cancel your order, please log into our portal and navigate to the 'Cancel Order' section.


You:  i have problem with my inverter


Bot: We've noted your issue: i have problem with my inverter. A technician will assist you soon.


...

Bot: It looks like you have a scheduled service, but I don’t have the details. Please check the portal.


You:  when will i get my order


Bot: To cancel your order, please log into our portal and navigate to the 'Cancel Order' section.


You:  exit


Goodbye!


In [32]:
import transformers
import torch
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers version: 4.51.3
Torch version: 2.7.0+cpu
CUDA available: False


In [33]:
train_set = set(train_texts)
val_set = set(val_texts)
overlap = train_set.intersection(val_set)
print("Overlap between train and val sets:", overlap)

Overlap between train and val sets: set()


In [19]:
print("Train set size:", len(train_texts))
print("Val set size:", len(val_texts))

Train set size: 804
Val set size: 202


In [36]:
# Check intent distribution in train and val sets
print("Train set intent distribution:")
print(train_data['intent'].value_counts())
print("Val set intent distribution:")
print(val_data['intent'].value_counts())

Train set intent distribution:
intent
resolved_query     40
ticket_query       40
cancel_order       40
scheduled_query    40
Name: count, dtype: int64
Val set intent distribution:
intent
cancel_order       10
scheduled_query    10
resolved_query     10
ticket_query       10
Name: count, dtype: int64
